In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json, os
import urllib.request
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence


In [ ]:

# ==========================================
# 0. Setup
# ==========================================

def drawing_to_stroke5(drawing, max_len=200):
    strokes = []
    for stroke in drawing:
        xs, ys = stroke[0], stroke[1]
        for i in range(len(xs)):
            dx = xs[i] - xs[i-1] if i > 0 else 0
            dy = ys[i] - ys[i-1] if i > 0 else 0
            if i < len(xs) - 1: p1, p2, p3 = 1, 0, 0
            else: p1, p2, p3 = 0, 1, 0
            strokes.append([dx, dy, p1, p2, p3])
    strokes.append([0, 0, 0, 0, 1])
    s5 = np.array(strokes, dtype=np.float32)
    if len(s5) > max_len:
        s5 = s5[:max_len]; s5[-1] = [0, 0, 0, 0, 1]
    return s5

def normalise_stroke5(stroke5):
    s = stroke5.copy()
    coords = s[:, :2]
    s[:, :2] = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
    return s

class QuickDrawDataset(Dataset):
    def __init__(self, path, max_len=200, max_samples=5000):
        self.samples = []
        with open(path) as f:
            for i, line in enumerate(f):
                if i >= max_samples: break
                d = json.loads(line)
                s5 = drawing_to_stroke5(d['drawing'], max_len=max_len)
                s5 = normalise_stroke5(s5)
                self.samples.append(torch.tensor(s5, dtype=torch.float32))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

def collate_fn(batch):
    lengths = [seq.shape[0] for seq in batch]
    padded  = pad_sequence(batch, batch_first=True, padding_value=0.0)
    return padded, lengths

os.makedirs('data', exist_ok=True)
path = 'data/cat.ndjson'
if not os.path.exists(path):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/quickdraw_dataset/full/simplified/cat.ndjson', path)

dataset = QuickDrawDataset(path, max_len=200, max_samples=3000)
loader  = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
print('Setup complete. Dataset size:', len(dataset))



In [ ]:

# ==========================================
# 1. The Encoder
# ==========================================

class Encoder(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.lstm      = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc_mu     = nn.Linear(hidden_dim * 2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim * 2, latent_dim)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        h = torch.cat([h[0], h[1]], dim=-1)
        return self.fc_mu(h), self.fc_logvar(h)

def reparameterise(mu, logvar):
    return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

encoder = Encoder(input_dim=5, hidden_dim=256, latent_dim=128)
padded, lengths = next(iter(loader))
mu, logvar = encoder(padded, lengths)
print('mu shape:', mu.shape)   # (batch, 128)



In [ ]:

# ==========================================
# 2. The Conductor
# ==========================================

class Conductor(nn.Module):
    def __init__(self, latent_dim=128, conductor_dim=512, output_dim=256, num_segments=16):
        super().__init__()
        self.num_segments = num_segments
        self.fc_init  = nn.Linear(latent_dim, conductor_dim)
        self.lstm     = nn.LSTM(output_dim, conductor_dim, batch_first=True)
        self.fc_out   = nn.Linear(conductor_dim, output_dim)
        self.output_dim = output_dim

    def forward(self, z):
        batch_size = z.shape[0]
        h = torch.tanh(self.fc_init(z)).unsqueeze(0)  # (1, batch, conductor_dim)
        c_cell = torch.zeros_like(h)

        input_c = torch.zeros(batch_size, 1, self.output_dim, device=z.device)
        c_list  = []

        for _ in range(self.num_segments):
            out, (h, c_cell) = self.lstm(input_c, (h, c_cell))
            c = self.fc_out(out)   # (batch, 1, output_dim)
            c_list.append(c)
            input_c = c

        return torch.cat(c_list, dim=1)

conductor = Conductor(latent_dim=128, conductor_dim=512, output_dim=256, num_segments=16)
z = reparameterise(mu, logvar)
c_seq = conductor(z)
print('Sub-goal sequence shape:', c_seq.shape)   # (batch, 16, 256)

# TODO: What does num_segments=16 mean if our sequences are max_len=200 steps?
#       How many steps does each sub-goal vector guide? Is this always a whole number?

# OBSERVATIONS:
# 1. Setting num_segments=16 means the entire timeline of the sketch is broken up into 16 macro-steps.
# 2. To find out how many fine-grained steps each sub-goal vector guides, we divide the batch sequence length 
#    by 16 (using `seq_len // 16 + 1`). This is not always a fixed, clean whole number; because sequences inside 
#    a batch vary up to 200 steps, the exact number of steps assigned to each sub-goal changes depending on the 
#    longest padded sample currently processed in that specific batch.



In [ ]:

# = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 
# 3. The Worker
# = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = 

class Worker(nn.Module):
    def __init__(self, input_dim=5, conductor_out_dim=256, worker_dim=512, output_dim=5):
        super().__init__()
        self.lstm   = nn.LSTM(input_dim + conductor_out_dim, worker_dim, batch_first=True)
        self.fc_out = nn.Linear(worker_dim, output_dim)

    def forward(self, stroke_seq, c_seq, segment_len):
        batch_size, seq_len, _ = stroke_seq.shape
        num_segments = c_seq.shape[1]

        c_expanded = c_seq.repeat_interleave(segment_len, dim=1)
        c_expanded = c_expanded[:, :seq_len, :]

        worker_input = torch.cat([stroke_seq, c_expanded], dim=-1)
        output, _ = self.lstm(worker_input)
        return self.fc_out(output)

worker = Worker(input_dim=5, conductor_out_dim=256, worker_dim=512, output_dim=5)
start_token   = torch.zeros(padded.shape[0], 1, 5)
decoder_input = torch.cat([start_token, padded[:, :-1, :]], dim=1)

segment_len = padded.shape[1] // conductor.num_segments + 1
output = worker(decoder_input, c_seq, segment_len)
print('Worker output shape:', output.shape)   # (batch, seq_len, 5)

# TODO: In the Worker, we concatenate the stroke step and sub-goal at every step.
#       Why is this better than only giving z to the decoder (as in the baseline VAE)?

# OBSERVATIONS:
# 1. In a baseline VAE, the initial bottleneck vector z is only used once to prime the initial state of the decoder. 
#    As the sequence grows longer (approaching 200 steps), the decoder RNN suffers from information decay or "forgetting" 
#    the global context, leading to distorted drawings.
# 2. Concatenating a dynamic sub-goal vector c_u at every single step forces a continuous structural blueprint into the 
#    Worker LSTM. The Conductor dynamically updates this blueprint every few steps, giving the network a systematic way 
#    to manage macro-objectives (like finishing an ear) before executing micro-movements (individual line segments).



In [ ]:

# ==========================================
# 4. Putting It Together: HierarchicalVAE
# ==========================================

class HierarchicalVAE(nn.Module):
    def __init__(self, input_dim=5, encoder_hidden=256, latent_dim=128, 
                 conductor_dim=512, conductor_out_dim=256, num_segments=16, worker_dim=512):
        super().__init__()
        self.num_segments     = num_segments
        self.conductor_out_dim = conductor_out_dim

        self.encoder   = Encoder(input_dim, encoder_hidden, latent_dim)
        self.conductor = Conductor(latent_dim, conductor_dim, conductor_out_dim, num_segments)
        self.worker    = Worker(input_dim, conductor_out_dim, worker_dim, input_dim)

    def forward(self, x, lengths):
        mu, logvar = self.encoder(x, lengths)
        z = reparameterise(mu, logvar)
        c_seq = self.conductor(z)

        start_token   = torch.zeros(x.shape[0], 1, x.shape[2], device=x.device)
        decoder_input = torch.cat([start_token, x[:, :-1, :]], dim=1)

        segment_len = x.shape[1] // self.num_segments + 1
        output = self.worker(decoder_input, c_seq, segment_len)
        return output, mu, logvar

model = HierarchicalVAE()
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total:,}')

out, mu, logvar = model(padded, lengths)
print('Output shape:', out.shape)   # (batch, seq_len, 5)



In [ ]:

# ==========================================
# 5. Loss and Training Loop
# ==========================================

def kl_loss(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

def hierarchical_loss(output, target, mu, logvar, kl_weight=0.5):
    recon_pos = F.mse_loss(output[:, :, :2], target[:, :, :2])
    pen_pred   = output[:, :, 2:]
    pen_target = target[:, :, 2:].argmax(dim=-1)
    recon_pen  = F.cross_entropy(
        pen_pred.reshape(-1, 3),
        pen_target.reshape(-1).long()
    )
    kl = kl_loss(mu, logvar)
    return recon_pos + recon_pen + kl_weight * kl, recon_pos, recon_pen, kl

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
model     = HierarchicalVAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20
history    = {'total': [], 'recon_pos': [], 'recon_pen': [], 'kl': []}

for epoch in range(num_epochs):
    model.train()
    totals = {k: 0.0 for k in history}
    kl_weight = min(0.5, epoch / num_epochs)

    for padded, lengths in loader:
        padded = padded.to(device)
        output, mu, logvar = model(padded, lengths)
        loss, rp, rpen, kl = hierarchical_loss(output, padded, mu, logvar, kl_weight)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        totals['total']     += loss.item()
        totals['recon_pos'] += rp.item()
        totals['recon_pen'] += rpen.item()
        totals['kl']        += kl.item()

    n = len(loader)
    for k in history: history[k].append(totals[k] / n)
    print(f'Epoch {epoch+1:02d}/{num_epochs} | '
          f'loss {history["total"][-1]:.4f} | '
          f'recon_pos {history["recon_pos"][-1]:.4f} | '
          f'kl {history["kl"][-1]:.4f}')

torch.save(model.state_dict(), 'hierarchical_vae.pth')
print('Model saved.')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(history['total']);     axes[0].set_title('Total loss')
axes[1].plot(history['recon_pos']); axes[1].set_title('Recon loss (position)')
axes[2].plot(history['kl']);        axes[2].set_title('KL loss')
for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout(); plt.show()



In [ ]:

# ==========================================
# 6. Visualising Reconstructions
# ==========================================

def stroke5_to_absolute(stroke5):
    abs_coords = np.cumsum(stroke5[:, :2], axis=0)
    pen_up = (stroke5[:, 3] + stroke5[:, 4]) > 0.5
    return abs_coords, pen_up

def plot_sketch5(stroke5_np, title='', ax=None):
    coords, pen_up = stroke5_to_absolute(stroke5_np)
    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 3))
    ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off'); ax.set_title(title, fontsize=9)
    start = 0
    for i in range(len(pen_up)):
        if pen_up[i]:
            seg = coords[start : i + 1]
            ax.plot(seg[:, 0], seg[:, 1], 'k-', linewidth=1.5)
            start = i + 1

model.eval()
with torch.no_grad():
    sample, lengths = next(iter(loader))
    sample = sample.to(device)
    recon, mu, logvar = model(sample, lengths)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    plot_sketch5(sample[i].cpu().numpy(),  title=f'Original {i}',      ax=axes[0, i])
    plot_sketch5(recon[i].cpu().numpy(),   title=f'Reconstructed {i}', ax=axes[1, i])
plt.suptitle('Top: original   Bottom: reconstructed', fontsize=11)
plt.tight_layout(); plt.show()

print('Checkpoint 3 complete -- Hierarchical Flow!')